# 06 - Quality Gate eğitimi
Görünürlük, poz güveni, görüş açısı ve geometrik tutarlılık özelliklerinden ağır hata olasılığı kalibre edilir. Eğitim/validation/test ayrımı kişi bazında olmalıdır.

In [ ]:
import sys, zipfile
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    artifacts_root = Path('/content/drive/MyDrive/PhysioLoad_AI_Project')
    code_root = Path('/content/physioload-ai')
    if not (code_root / 'physioload_env.py').exists():
        code_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(artifacts_root / 'code' / 'physioload-source.zip') as package: package.extractall(code_root)
else:
    artifacts_root = None
    here = Path.cwd().resolve()
    code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))
from physioload_env import setup_environment
env = setup_environment(root=code_root, artifacts_root=artifacts_root)
print(env.summary())

In [ ]:
import joblib, numpy as np, pandas as pd
from physioload.metrics import classification_auc, risk_coverage
from physioload.quality import fit_calibrator

training_path = env.predictions_dir / 'quality_training_rows.csv'
feature_columns = ['visibility','pose_confidence','view_deviation_deg','geometry_penalty','scale_valid','load_definition_valid']
target_column = 'severe_error'
if not training_path.exists():
    raise FileNotFoundError(f'Önce referanslı tahmin satırlarını üretin: {training_path}')
table = pd.read_csv(training_path)
missing = set(feature_columns + [target_column, 'split']) - set(table.columns)
if missing:
    raise ValueError(f'Eksik sütunlar: {sorted(missing)}')
train = table[table.split == 'train']
validation = table[table.split == 'validation']
test = table[table.split == 'test']
model = fit_calibrator(train[feature_columns].to_numpy(), train[target_column].to_numpy())
print('train:', len(train), 'validation:', len(validation), 'test:', len(test))

In [ ]:
# Eşik yalnız validation'da seçilir; test nihai raporda bir kez kullanılır.
validation_probability = model.predict_proba(validation[feature_columns])[:, 1]
validation_metrics = classification_auc(validation[target_column], validation_probability)
validation_metrics

In [ ]:
model_path = env.calibrators_dir / f'quality_gate_{env.processed_version}.joblib'
joblib.dump({'model': model, 'features': feature_columns, 'processed_version': env.processed_version}, model_path)
print('saved:', model_path)